# Multi-Agent System

A collaborative AI system where four specialised agents work together to answer queries
using Sarvam `sarvam-m`.

## Architecture

```
User Query
        |
        v
Planner Agent       <- sarvam-m: breaks query into tasks (JSON)
        |
        v
Researcher Agent    <- search tool + sarvam-m synthesis
Executor Agent      <- calculator tool + sarvam-m fallback
        |
        v
Critic Agent        <- sarvam-m: reviews results, writes final answer
        |
        v
Final Answer
```

In [ ]:
%pip install sarvamai>=0.1.26 python-dotenv>=1.0.0

## Setup

Load environment variables, initialise the Sarvam AI client, and configure `sys.path`
so that `agents/` and `tools/` are importable from the notebook.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from sarvamai import SarvamAI

load_dotenv()

SARVAM_API_KEY = os.environ.get("SARVAM_API_KEY", "")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "SARVAM_API_KEY is not set. "
        "Copy .env.example to .env and paste your key."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)

# Add recipe root to sys.path so agents/ and tools/ are importable
_RECIPE_ROOT = str(Path.cwd())
if _RECIPE_ROOT not in sys.path:
    sys.path.insert(0, _RECIPE_ROOT)

from agents.planner import plan
from agents.researcher import research
from agents.executor import execute_task
from agents.critic import critique

print("Setup complete.")

## Tools

The agents rely on two stdlib-backed tools. Quick verification that both work correctly.

In [ ]:
from tools.calculator import calculate
from tools.search import search

print("calculator('12 * 8')  ->", calculate("12 * 8"))
print()
print("search('Sarvam AI')   ->")
print(search("Sarvam AI"))

## Agents

### Planner

`plan` sends the query to `sarvam-m` and receives a JSON task list. Each task specifies
which agent should handle it and what input to pass.

In [ ]:
sample_query = "What is 20 percent of 500, and what is Sarvam AI?"
tasks = plan(sample_query, client)

print("Query:", sample_query)
print()
print("Plan:")
print(json.dumps(tasks, indent=2))

### Researcher and Executor

`research` retrieves and summarises information using the search tool and `sarvam-m`.
`execute_task` runs arithmetic first; falls back to `sarvam-m` for non-numeric inputs.

In [ ]:
if tasks:
    for task in tasks[:2]:  # show first two tasks individually
        if task["agent"] == "executor":
            result = execute_task(task, client)
            print(f"Executor result for task {task['task_id']}:")
        else:
            result = research(task, client)
            print(f"Researcher result for task {task['task_id']}:")
        print(result)
        print()

## Orchestrator

`orchestrate` chains all four agents into a single call:
1. Planner decomposes the query into tasks.
2. Researcher and Executor handle their assigned tasks in order.
3. Critic reviews all results and produces the final answer.

In [ ]:
def orchestrate(
    query: str,
    sarvam_client: SarvamAI = client,
) -> dict[str, Any]:
    """Run the full multi-agent pipeline.

    Phases:
        1. Plan   -- Planner agent decomposes query into tasks.
        2. Execute -- Researcher and Executor agents handle their tasks.
        3. Critique -- Critic agent reviews all results.

    Args:
        query: User task or question.
        sarvam_client: Initialised SarvamAI client.

    Returns:
        Dict with keys: query, tasks, results, final_answer.
    """
    # Phase 1: Plan
    tasks = plan(query, sarvam_client)

    # Phase 2: Execute researcher and executor tasks
    results: list[dict[str, Any]] = []
    for task in tasks:
        agent_name = task.get("agent", "researcher")
        if agent_name == "executor":
            result = execute_task(task, sarvam_client)
        else:
            result = research(task, sarvam_client)
        results.append({"task": task, "result": result})

    # Phase 3: Critique
    final_answer = critique(query, results, sarvam_client)

    return {
        "query": query,
        "tasks": tasks,
        "results": results,
        "final_answer": final_answer,
    }

## Demo 1 — Calculation and Knowledge

Run the orchestrator on a query that requires both a numeric calculation and a
factual lookup.

In [ ]:
output = orchestrate(
    "What is 20 percent of 500, and what is Sarvam AI?",
    sarvam_client=client,
)

print("Query:", output["query"])
print()
print("Tasks assigned:")
for task in output["tasks"]:
    print(f"  Task {task['task_id']} -> {task['agent']}: {task['description']}")
print()
print("Step results:")
for item in output["results"]:
    print(f"  Task {item['task']['task_id']}: {item['result']}")
print()
print("Final answer:")
print(output["final_answer"])

## Demo 2 — Pure Knowledge Query

Run the orchestrator on a query that only requires research.

In [ ]:
output2 = orchestrate(
    "What are AI agents and how does LangChain relate to them?",
    sarvam_client=client,
)

print("Query:", output2["query"])
print()
print("Tasks assigned:")
for task in output2["tasks"]:
    print(f"  Task {task['task_id']} -> {task['agent']}: {task['description']}")
print()
print("Final answer:")
print(output2["final_answer"])

## Extending the System

To add a new agent:

1. Create `agents/my_agent.py` with a single entry-point function.
2. Import it in cell 3 alongside the existing imports.
3. Register it in `orchestrate()` with an `elif agent_name == "my_agent":` branch.
4. Add `"my_agent"` as a valid value in `agents/planner.py`'s `_SYSTEM_PROMPT`.

In [ ]:
# Example: a summariser agent that condenses any text
def summarise(task: dict[str, Any], sarvam_client: SarvamAI = client) -> str:
    """Example custom agent: summarise the input text using sarvam-m."""
    messages = [
        {
            "role": "system",
            "content": "Summarise the following text in one sentence.",
        },
        {"role": "user", "content": task.get("input", "")},
    ]
    response = sarvam_client.chat.completions(messages=messages, model="sarvam-m")
    return response.choices[0].message.content.strip()


# Register it inline for demonstration
test_task = {
    "task_id": 99,
    "agent": "summariser",
    "description": "Summarise what RAG is",
    "input": (
        "Retrieval-Augmented Generation (RAG) combines a retrieval step "
        "with a language model to answer questions grounded in a document corpus."
    ),
}
print("Custom agent result:", summarise(test_task, client))

## Error Reference and Resources

### Common Errors

| Error | Cause | Fix |
|:------|:------|:----|
| `RuntimeError: SARVAM_API_KEY is not set` | Missing `.env` | Copy `.env.example` to `.env` and add your key |
| `sarvamai.APIStatusError: 401` | Invalid API key | Verify key at dashboard.sarvam.ai |
| `sarvamai.APIStatusError: 429` | Rate limit exceeded | Add a short delay between agent calls |
| `ModuleNotFoundError: agents` | `sys.path` not configured | Ensure cell 3 ran before other cells |
| `json.JSONDecodeError` in planner | Non-JSON response | Handled -- falls back to single researcher task |

### Resources

- [Sarvam AI Documentation](https://docs.sarvam.ai/)
- [Sarvam API Dashboard](https://dashboard.sarvam.ai/)